In [1]:
import pandas as pd
import numpy as np
from tslearn.clustering import TimeSeriesKMeans
import matplotlib.pyplot as plt
from utils import *

In [2]:
df = pd.read_excel('Experiment/2_hTERT_HME1/Data/Processed/Full_dataset_2_hTERT_HME1_functional_names_diff_scaled.xlsx')
# df = pd.read_excel('Experiment/1_HEK293T/Data/Processed/Full_dataset_HEK293T.xlsx')
# df = pd.read_excel('Experiment/1_hTERT_HME1/Data/Processed/Handmade_Log2_FC_from_FGZC.xlsx')
df = df.fillna(0)
df

,protein_Id,description,protein_name,Peptide,MaxPepProbability,site_start,site_end,n_sites,localized_sites,phospho_sites,...,FC_scaled_INS_10,FC_scaled_INS_15,FC_scaled_INS_90,FC_scaled_EGFnINS_full,FC_scaled_EGFnINS_starve,FC_scaled_EGFnINS_2,FC_scaled_EGFnINS_5,FC_scaled_EGFnINS_10,FC_scaled_EGFnINS_15,FC_scaled_EGFnINS_90
0,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,SGSGNFGGGR,0.9956,197,199,1,0,0,...,0.242030,0.681547,0.363844,1.000000,0,-0.035173,0.090278,0.206244,0.684762,0.659816
1,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,SGsGNFGGGR,1.0000,197,199,1,1,S199,...,0.213350,0.612315,0.650989,1.000000,0,-0.049035,0.167233,0.418818,0.585366,0.779797
2,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,SKSEsPKEPEQLR,1.0000,2,6,1,1,S6,...,-0.455567,-0.395045,-0.875492,-0.186823,0,-0.673611,-0.487599,-0.781955,-0.057160,-0.119301
3,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,sKsESPK,0.9894,2,6,2,2,S2;S4,...,-0.105284,-0.716049,-0.203130,-0.922523,0,-1.000000,-0.178269,-0.869578,-0.794457,-0.810823
4,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,NQGGYGGSSSSSSYGSGR;NQGGYGGSSSSSSYGSGRRF,1.0000,305,316,1,0,0,...,-0.402769,0.529268,0.727333,-0.440162,0,-0.248076,0.255346,0.676671,0.976998,0.341977
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50009,rev_Q5VWT5,0,0,GALMNRRGtHVNtMQIK,0.9898,303,307,2,2,T303;T307,...,0.254396,0.556799,0.358769,1.000000,0,-0.107027,0.072819,0.713544,0.419371,0.830938
50010,rev_Q8N6M0,0,0,LQDEIAKYMCHGDsPIQK,0.9970,135,141,1,1,S141,...,0.530347,0.367762,0.632670,0.049028,0,0.348857,0.520648,0.721462,0.458464,0.500749
50011,rev_Q96M83,0,0,ERGPtIKK,0.9835,505,505,1,1,T505,...,0.685330,0.492152,0.779448,0.047561,0,0.125190,0.618619,1.000000,0.823704,0.792848
50012,rev_Q9ULL1,0,0,SSASSSTSGFSVPR,0.9787,1366,1376,2,0,0,...,0.262822,0.291193,-0.115279,0.086867,0,-0.086244,1.000000,0.257387,0.066917,-0.174938


In [3]:
# Filtering only "FC_" columns, excluding those with "statistics"

data_type = "log2_FC" #data type over which I want to do the filtering and the clustering

column_names = df.columns.tolist()
column_selection = [element for element in column_names if f"{data_type}" in element]

# df = df.loc[df["localized_sites"] > 0]
# df = df.loc[df["n_rep"] > 1]

# Apply filtering condition on selected columns
subset = df[column_selection]
# subset = subset.head(5).copy()
# subset

# Make sub dataframes dividing based on data being in or out a bracket of values
row_max = subset.max(axis=1)
row_min = subset.min(axis=1)

mask_extremes = (row_max > 0.5) | (row_min < -0.5)

# Step 4: Filter rows into two DataFrames
df_over_05 = df[mask_extremes]
df_below_05 = df[~mask_extremes]

print(len(df), len(df_over_05), len(df_below_05), (len(df_over_05)+len(df_below_05)))

50014 8258 41756 50014


In [4]:
#Clustering based on the data_type
data_type = "log2_FC"

column_names = df_over_05.columns.tolist()
column_selection = [element for element in column_names if f"{data_type}" in element]

multivariate_df, names_of_myseries = reshape_df(df = df_over_05, time_series = column_selection, dimensions = 3, len_time_serie = 7)
print(f"\nThe size of the dataset is {multivariate_df.shape}")

clustering = TimeSeriesKMeans(n_clusters=9, max_iter=5000, metric='euclidean', max_iter_barycenter=1000, verbose=True, random_state=0).fit(multivariate_df)

df_over_05[f"{data_type}_clusters"] = clustering.labels_



The size of the dataset is (8258, 3, 7)
1.870 --> 1.524 --> 1.492 --> 1.477 --> 1.470 --> 1.466 --> 1.464 --> 1.463 --> 1.463 --> 1.462 --> 1.462 --> 1.462 --> 1.461 --> 1.459 --> 1.458 --> 1.456 --> 1.453 --> 1.452 --> 1.452 --> 1.452 --> 1.451 --> 1.451 --> 1.451 --> 1.451 --> 1.451 --> 1.451 --> 1.451 --> 1.451 --> 1.451 --> 1.451 --> 1.451 --> 1.451 --> 


/var/folders/jj/dfknz6j944d69gcp558vh1j80000gn/T/ipykernel_32096/1734196584.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_over_05[f"{data_type}_clusters"] = clustering.labels_


In [6]:
# df.to_excel("Experiment/1_hTERT_HME1/Results/tslearn_Clusters/25_clusters/1_hTERT_HME1_Full_dataset_25_clusters.xlsx", index=False)
# df = pd.read_excel("Experiment/1_hTERT_HME1/Results/tslearn_Clusters/25_clusters/1_hTERT_HME1_Full_dataset_25_clusters.xlsx")


,protein_Id,description,protein_name,Peptide,MaxPepProbability,site_start,site_end,n_sites,localized_sites,phospho_sites,...,FC_scaled_INS_15,FC_scaled_INS_90,FC_scaled_EGFnINS_full,FC_scaled_EGFnINS_starve,FC_scaled_EGFnINS_2,FC_scaled_EGFnINS_5,FC_scaled_EGFnINS_10,FC_scaled_EGFnINS_15,FC_scaled_EGFnINS_90,log2_FC_clusters
1,A0A2R8Y4L2,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,SGsGNFGGGR,1.0000,197,199,1,1,S199,...,0.612315,0.650989,1.000000,0,-0.049035,0.167233,0.418818,0.585366,0.779797,5
15,A0AVK6,Transcription factor E2F8 OS=Homo sapiens OX=9...,E2F8,HPsLIK,0.9753,396,396,1,1,S396,...,-0.403172,-0.119672,-0.763842,0,0.072489,-0.034891,-0.071995,-1.000000,-0.778334,1
21,A0FGR8,Extended synaptotagmin-2 OS=Homo sapiens OX=96...,ESYT2,RPsVSK,0.9849,676,678,1,1,S676,...,0.774689,0.682207,1.000000,0,0.136687,0.534376,0.728067,0.927492,0.911908,5
29,A0FGR8,Extended synaptotagmin-2 OS=Homo sapiens OX=96...,ESYT2,SSSsLLAsPGHISVK,1.0000,736,748,2,2,S739;S743,...,-0.922820,-0.747518,-0.315211,0,0.197256,-0.782070,-0.494545,-0.947546,-1.000000,1
38,A0FGR8,Extended synaptotagmin-2 OS=Homo sapiens OX=96...,ESYT2,APsPPR,0.9619,91,91,1,1,S91,...,-0.611934,-0.365238,-0.481972,0,-0.916966,-0.101934,-0.680156,-0.227201,-0.758305,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49746,Q9Y3Y2,Chromatin target of PRMT1 protein OS=Homo sapi...,CHTOP,GHLDAELDAYMAQtDPETND,1.0000,238,246,1,1,T242,...,-0.686894,-0.056693,-0.491929,0,-0.792870,-0.904641,0.075308,-0.708776,-0.928327,1
49750,Q9Y4F4,TOG array regulator of axonemal microtubules p...,TOGARAM1,SFEGLSKPsPQKK,0.9985,855,863,1,1,S863,...,-0.402551,-0.283698,-0.322670,0,0.250540,-0.796420,-0.847879,-1.000000,-0.433808,1
49753,Q9Y576,Ankyrin repeat and SOCS box protein 1 OS=Homo ...,ASB1,AGPGsAGR,0.9898,15,15,1,1,S15,...,-0.208112,-0.449528,-0.511079,0,-0.135657,-0.478919,0.200528,0.195496,-0.173449,1
49758,Q9Y676,Small ribosomal subunit protein mS40 OS=Homo s...,MRPS18B,TPAEASSTGQTGPQSAL,1.0000,242,256,1,0,0,...,1.000000,0.484239,0.709438,0,0.449736,0.566802,0.955856,0.671097,0.719251,0


In [7]:
clusters_plot(df=df_over_05, legend=['4.68 nM EGF', '10 nM Ins', '4.68 nM EGF + 10 nM Ins'],
              saving_path="",
              cluster_column = "log2_FC_clusters", data_type = "log2_FC", plot_different_data = True,
              cluster_name="testing Function", saving_info="hola que tal",
              y_lims_list=False,
              save_pdf=False, save_png=False, plot_close=True)

testing Functionhola que tal Plot not saved


In [9]:
# df = pd.read_excel('Experiment/1_hTERT_HME1/Results/Clusters/Cluster6/Full_dataset_NORMALIZED_DATA_clustering.xlsx')
# df_over_05_test = df_over_05.head(100).copy()

plot_dataset_phosphosites(df = df_over_05_test, cluster_column = "log2_FC_clusters", cluster_number = 4,
                          data_type = "log2_FC",
                          legend=['4.68 nM EGF', '10 nM Ins', '4.68 nM EGF + 10 nM Ins'], color_palette = ['r', 'b', 'fuchsia'],
                          saving_path="",
                          dataset_name="", saving_info="",
                          plot_individually=False,
                          fit_y_lims=False,
                          save_pdf=False, save_png=False, plot_close=True)

log2_FC_clusters_group4_ Plot not saved


/Users/ignacionavascamacho/PycharmProjects/TMT_Data_analysis/utils.py:290: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.sort_values(by=['site'], inplace=True)
